# Machine Learning: Telecom Customer Churn Prediction and Business Insights

## Install and import the required libraries :

In [ ]:
#!pip install yellowbrick
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, RepeatedKFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import BaggingClassifier, BaggingRegressor, StackingClassifier, IsolationForest
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn import metrics
from sklearn.metrics import roc_curve, auc, recall_score, f1_score
from sklearn.metrics import accuracy_score
import xgboost
from xgboost import XGBClassifier, XGBRegressor
#import warnings
#warnings.filterwarnings('ignore')

## Import the dataset

In [ ]:
data = pd.read_csv('telecom_churn.csv')
df = data.copy()

## 1. Inspect the data

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.info()

## 2. Data preprocessing (preliminary)

In [ ]:
#delete the useless columns
col = ['account length', 'phone number']
df.drop(col, axis=1, inplace=True)

#Transform the value and data type
regions = {
    'Northeast':['ME', 'VT', 'NH', 'MA', 'RI', 'CT', 'NY', 'NJ', 'DC', 'PA'],
    'Midwest':['OH', 'IN', 'IL', 'MI', 'WI', 'MN', 'IA', 'MO', 'KS', 'NE', 'SD', 'ND'],
    'Southeast':['AL', 'FL', 'GA', 'SC', 'NC', 'VA', 'WV', 'MD', 'DE', 'MS', 'TN'],
    'Southwest':['AZ', 'NM', 'OK', 'TX', 'AR', 'LA', 'KY'],
    'West':['AK', 'CA', 'CO', 'HI', 'ID', 'MT', 'NV', 'OR', 'UT', 'WA', 'WY']
}
for i in regions.keys():
    df.loc[df['state'].isin(regions[i]), 'state'] = i
df.rename(columns = {'state':'region'}, inplace = True) 
Counter(df['region'])

In [ ]:
'''
San Francisco: 415
Oakland: 510
San Jose: 408
'''
df['area code'] = df['area code'].astype(object)
df.info()

## 3. Exploratory Data Analysis, EDA:

In [ ]:
label = df['churn'].value_counts().keys()
x = df['churn'].value_counts().values
plt.pie(x, labels=label, autopct='%.2f%%')
plt.title('Churn Ratio')
plt.show() 

In [ ]:
categorical_columns = [i for i in df.columns if (df[i].dtype == 'object') | ((df[i].dtype == 'bool'))]
categorical_columns

plt.figure(figsize=[10, 12])
for idx, col in enumerate(df.loc[:, categorical_columns].columns, start=1):
    plt.subplot(3, 2, idx)
    sns.countplot(data=df.loc[:, categorical_columns], x=col, hue='churn')
    plt.legend(['Churn(False)', 'Churn(True)'], fontsize=8)
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
for col in categorical_columns[:-1]:
    num = df[col].value_counts()
    deno = df.loc[df['churn']==True, col].value_counts()
    ratio = pd.DataFrame({'Churn Rate': deno / num})
    display(ratio)

In [ ]:
sns.catplot(data=df, x='region', y='churn', kind='bar', hue='area code')
plt.xlabel('Region')
plt.ylabel('Churn')
plt.show()

In [ ]:
num_columns = [i for i in df.columns if (df[i].dtype != 'object') & ((df[i].dtype != 'bool'))]
plt.figure(figsize=(15,12))
sns.heatmap(df.loc[:, num_columns].corr() , annot =True)
plt.show()

In [ ]:
col = ['total day minutes', 'total eve minutes', 'total night minutes', 'total intl minutes']
df.drop(col, axis=1, inplace=True)
num_columns = [i for i in df.columns if (df[i].dtype != 'object') & ((df[i].dtype != 'bool'))]

In [ ]:
#numeric variables
plt.figure(figsize=[10, 12])
for idx, col in enumerate(df.loc[:, num_columns].columns, start=1):
    plt.subplot(5, 3, idx)
    plt.hist(x=col, data=df, bins=50)
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=[20, 36])
for idx, col in enumerate(df.loc[:, num_columns].columns, start=1):
    plt.subplot(5, 3, idx)
    sns.boxplot(x='churn',y=col, data = df, hue='churn')
    plt.title(col)
    plt.legend(loc='upper right', title='Churn')
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(data=df, vars=num_columns, hue='churn')
plt.show()

## 4. Data preprocessing

In [ ]:
#transform the categorical data
for i in categorical_columns[:-1]:
    encoder = LabelEncoder()
    data_encoded = encoder.fit_transform(df[i])
    df[i] = data_encoded
for j in categorical_columns[:-1]:   #check up on the values
    print(Counter(df[j]))

#split the data into training set and testing set
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:-1], df.iloc[:,-1], 
                                                    test_size=0.2,
                                                    shuffle=True,
                                                    random_state=42)
#eliminate the outliers in the training data
print(f"Number of raw data: {len(X_train)}")
detector = IsolationForest(n_estimators=1000,
                           contamination=0.05,
                           random_state=42,
                           n_jobs=-1)
labels = detector.fit_predict(X_train.loc[:, num_columns])
X_train.loc[:, 'labels'] = labels
X_train = X_train.loc[X_train['labels']==1, :]
X_train.drop('labels', axis=1, inplace=True)
y_train = pd.DataFrame(y_train)
y_train.loc[:, 'labels'] = labels
y_train = y_train.loc[y_train['labels']==1, :]
y_train.drop('labels', axis=1, inplace=True)
y_train = y_train['churn']
print(f"Number of data after processing: {len(X_train)}")

#scaled the data
scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

## balance the data based on SMOTE method

In [ ]:
y_train.value_counts()

In [ ]:
smote = SMOTE(sampling_strategy = "auto",
              random_state = 42,
              k_neighbors = 5)
X_smote, y_smote = smote.fit_resample(X_train_scaled, y_train)
Counter(y_smote)

## 5. Modeling:

In [ ]:
### (1) K-means

In [ ]:
df_scaler = MinMaxScaler(feature_range=(0, 1))
df_scaled = df_scaler.fit_transform(df.loc[: ,num_columns])
df_scaled.shape

k=8
model = KMeans(random_state=42, max_iter=500)
visulizer = KElbowVisualizer(model, k=range(1, k+1))
visulizer.fit(df_scaled)
plt.xticks(range(1, k+1))
visulizer.show()

In [ ]:
model = KMeans(n_clusters=3, random_state=42, max_iter=500)
label = model.fit_predict(df_scaled)
df['cluster'] = label
Counter(df['cluster'])

In [ ]:
cluster_info = df.groupby('cluster').mean()
cluster_info.loc[:,num_columns]

In [ ]:
cluster_info_t = np.transpose(cluster_info.loc[:, num_columns])
columns = [
    ['number vmail messages'],
    ['total day calls', 'total eve calls', 'total night calls'],
    ['customer service calls'],
]
plt.figure(figsize=(10, 4))
for idx, col in enumerate(columns, start=1):
    plt.subplot(1, 3, idx)
    plt.plot(cluster_info_t.loc[col], marker='.', markersize=10, linestyle=None)
    plt.legend(['Cluster 1', 'Cluster 2', 'Cluster 3'], loc='lower left', fontsize=6)
    plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### (2) Principle Component Analysis, PCA

In [ ]:
pca = PCA(n_components=None)
pca.fit(df_scaled)
each = pca.explained_variance_ratio_
pca_number = [i for i in range(1, len(each)+1)]
cum = pca.explained_variance_ratio_.cumsum()
pca_table = pd.DataFrame({'Principal Components': pca_number,
                          '個別主成份解釋變異比例': each,
                          '累計解釋變異比例': cum})
pca_table.set_index('Principal Components', inplace=True)
pca_table

In [ ]:
plt.figure(figsize=[8, 6])
plt.plot(range(1, df_scaled.shape[1]+1), pca.explained_variance_ratio_.cumsum(), 
         linestyle="-", linewidth=1, marker=".", color="blue")
plt.xlabel("Principal Component")
plt.ylabel("explained variance ratio cumsum")
plt.title("Principal Component Analysis")
plt.grid(True)
plt.xticks(ticks=range(1, 10+1), labels=["PC"+str(n) for n in range(1, 10+1)], fontsize=8.5)
plt.axhline(y = cum[6],    #取>80%
            xmin = 0, xmax = 1, linestyle="--", linewidth=1, color="red")
plt.axvline(x = 7, 
            ymin = 0, ymax = 1, linestyle="--", linewidth=1, color="red")

plt.show()

In [ ]:
pca = PCA(n_components=2)
pca_2d = pca.fit_transform(df_scaled)
df.loc[:, 'pc1'] = pca_2d[:, 0]; df.loc[:, 'pc2'] = pca_2d[:, 1]

plt.figure(figsize=[10, 3])
sns.heatmap(pca.components_, cmap='viridis', 
            xticklabels=num_columns, yticklabels=['PC 1', 'PC 2'], annot=True)

In [ ]:
plt.figure(figsize=[10, 4])
plt.subplot(1, 2, 1)
sns.scatterplot(data=df, x='pc1', y='pc2', hue='cluster', s=15)
plt.xlabel('PC 1'); plt.ylabel('PC 2')
plt.legend(['Cluster 1', 'Cluster 2', 'Cluster 3'])
plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x='pc1', y='pc2', hue='churn', s=15)
plt.xlabel('PC 1'); plt.ylabel('PC 2')
plt.tight_layout()
plt.show()

### (3) t-SNE

In [ ]:
tsne = TSNE(n_components=2, n_iter=1000, perplexity=20)
X_new = tsne.fit_transform(df_scaled)

In [ ]:
plt.figure(figsize=[8, 3])
plt.subplot(1, 2, 1)
sns.scatterplot(x=X_new[:,0], y=X_new[:, 1], hue=label, s=10)
plt.legend(['Cluster 1', 'Cluster 2', 'Cluster 3'], loc='upper right', fontsize=8)
plt.subplot(1, 2, 2)
sns.scatterplot(x=X_new[:,0], y=X_new[:, 1], hue=df['churn'], s=10)
plt.legend(['Churn(False)', 'Churn(True)'], loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## (4) Gaussian Naive Bayes

In [ ]:
def combo_results(X_tr=X_train_scaled, y_tr=y_train,
                  X_te=X_test_scaled, y_te=y_test,
                  X_smo=X_smote, y_smo=y_smote,
                  feature_imp=False):
    '''being used after initializing the model'''
    #raw data
    raw_model = model.fit(X_tr, y_tr)
    train_accuracy = accuracy_score(y_tr, raw_model.predict(X_tr))
    y_te_pred = raw_model.predict(X_te)
    test_accuracy = accuracy_score(y_te, y_te_pred) 
    test_f1 = f1_score(y_te, y_te_pred, zero_division=0)
    test_sensitivity = recall_score(y_te, y_te_pred, zero_division=0)
    raw_prob = raw_model.predict_proba(X_te)[:, 1]
        
    #smote data
    smo_model = model.fit(X_smo, y_smo)
    train_smo_accuracy = accuracy_score(y_smo, smo_model.predict(X_smo))
    y_smo_te_pred = smo_model.predict(X_te)
    test_smo_accuracy = accuracy_score(y_te, y_smo_te_pred)
    test_smo_f1 = f1_score(y_te, y_smo_te_pred, zero_division=0)
    test_smo_sensitivity = recall_score(y_te, y_smo_te_pred, zero_division=0)
    smo_prob = smo_model.predict_proba(X_te)[:, 1]

    #show the comparison details
    data = [['Raw Data', train_accuracy, test_accuracy, test_f1, test_sensitivity],
           ['SMOTE Data', train_smo_accuracy, test_smo_accuracy, test_smo_f1, test_smo_sensitivity]]
    display(pd.DataFrame(data, columns=['Data Type', 'Training Accuracy', 'Testing Accuracy', 'Testing F1 Score', 'Testing Sensitivity']).set_index('Data Type'))

    #Confusion Matrix
    params = [
        [metrics.confusion_matrix(y_test, y_te_pred), ' (Raw Data)'],
        [metrics.confusion_matrix(y_test, y_smo_te_pred), ' (SMOTE)'],
    ]
    plt.figure(figsize=[12, 5])
    for idx, param in enumerate(params, start=1):
            
        plt.subplot(1, 2, idx)
        sns.heatmap(param[0], annot=True, fmt='d', cmap='Blues',
                    xticklabels=[False, True], yticklabels=[False, True])
        plt.xlabel('Predicted Labels')
        plt.ylabel('Actual Labels')
        plt.title('Confusion Matrix of Testing Results' + param[1])
    plt.tight_layout()
    plt.show()
       
    #ROC, AUC
    raw_fpr, raw_tpr, _ = roc_curve(y_true=y_te, y_score=raw_prob)  #raw data
    smo_fpr, smo_tpr, _ = roc_curve(y_true=y_te, y_score=smo_prob)  #SMOTE
    params = [
        [raw_fpr, raw_tpr, auc(raw_fpr, raw_tpr), ' (Raw Data)'],
        [smo_fpr, smo_tpr, auc(smo_fpr, smo_tpr), ' (SMOTE)'],
    ]
    plt.figure(figsize=[12, 5])
    for idx, param in enumerate(params, start=1):
        plt.subplot(1, 2, idx)
        plt.plot(param[0], param[1], color="red", linestyle="--", linewidth=1, label=f"True : AUC={param[2]:.4f}")
        plt.plot((0,1),(0,1), color="gray", linestyle="-", linewidth=1)
        plt.xlim(0, 1); plt.ylim(0, 1)
        plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title("ROC Curve for Testing Dataset" + param[3])
        plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    #feature_importance
    if feature_imp:
        f_imp_raw = pd.Series(raw_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
        plt.figure(figsize=[8, 6])
        plt.subplot(2, 1, 1)
        sns.barplot(x=f_imp_raw, y=X_train.columns)
        plt.xlabel('Feature Importance (Raw Data)'); plt.ylabel(None)
        
        f_imp_smo = pd.Series(smo_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
        plt.subplot(2, 1, 2)
        sns.barplot(x=f_imp_smo, y=X_train.columns)
        plt.xlabel('Feature Importance (SMOTE)'); plt.ylabel(None)
        plt.tight_layout(); plt.show()     
    
    return train_accuracy, test_accuracy, test_f1, test_sensitivity, train_smo_accuracy, test_smo_accuracy, test_smo_f1, test_smo_sensitivity

In [ ]:
model = GaussianNB()
model.fit(X_train_scaled, y_train)
gaussian_nb_res = combo_results()

### (5) Logistic Regression

In [ ]:
#create a dataframe only for logistic regression
#since we are going to use one hot encoding for our logistic regression model
X_logistic_train = X_train.copy()
X_logistic_train[num_columns] = X_logistic_train[num_columns].astype(float)
X_logistic_train.loc[:, num_columns] = X_train_scaled[:, 4:]
X_logistic_train[categorical_columns[:-1]] = X_logistic_train[categorical_columns[:-1]].astype(object)
X_logistic_train = pd.get_dummies(X_logistic_train, drop_first=True)

X_logistic_test = X_test.copy()
X_logistic_test[num_columns] = X_logistic_test[num_columns].astype(float)
X_logistic_test.loc[:, num_columns] = X_test_scaled[:, 4:]
X_logistic_test[categorical_columns[:-1]] = X_logistic_test[categorical_columns[:-1]].astype(object)
X_logistic_test = pd.get_dummies(X_logistic_test, drop_first=True)

smote_logistic = SMOTE(sampling_strategy = "auto",
              random_state = 42,
              k_neighbors = 5)
X_smote_logistic, y_smote_logistic = smote_logistic.fit_resample(X_logistic_train, y_train)
print(Counter(y_smote_logistic))

In [ ]:
model = LogisticRegression(solver="lbfgs", max_iter=1000, random_state=42)
logistic_res = combo_results(X_tr=X_logistic_train, y_tr=y_train,
                             X_te=X_logistic_test, y_te=y_test,
                             X_smo=X_smote_logistic, y_smo=y_smote_logistic)

### (6) kNN

In [ ]:
param_grid = {"n_neighbors":[i for i in range(1, 20+1)]}
searchCV = GridSearchCV(KNeighborsClassifier(), param_grid, scoring="recall",
                        verbose=3, cv=10, refit=True, n_jobs=-1)
best_model = searchCV.fit(X_train_scaled, y_train)
print(f'The Best Hyparameters: {best_model.best_params_}')

In [ ]:
model = KNeighborsClassifier(n_neighbors=1)
knn_res = combo_results()

### (7) Decision Tree

In [ ]:
model = DecisionTreeClassifier(criterion='gini', max_depth=None, random_state=42)
dt_res = combo_results(feature_imp=True)

### (7) Decision Tree (Bagging)

In [ ]:
%%time
model = BaggingClassifier(estimator=DecisionTreeClassifier(criterion='gini'),
                          n_estimators=1000, bootstrap=True, oob_score=True,
                          n_jobs=-1, random_state=42)
dt_bagging_res = combo_results(feature_imp=False)

### (8) Random Forest (Bagging)

In [ ]:
%%time
model = RandomForestClassifier(n_estimators=1000,criterion='gini',
                               bootstrap=True, oob_score=True,
                               n_jobs=-1, random_state=42)
rf_res = combo_results(feature_imp=True)

### (9) ADA Boost

In [ ]:
%%time
param_grid = {"learning_rate":[i/10 for i in range(1, 11)]}             
searchCV = RandomizedSearchCV(AdaBoostClassifier(n_estimators=1000, algorithm='SAMME', random_state=42), 
                              param_grid, scoring="recall", cv=10, refit=True, n_jobs=-1)
best_model = searchCV.fit(X_train_scaled, y_train)
print(f'The Best Hyparameters: {best_model.best_params_}')

In [ ]:
%%time
model = AdaBoostClassifier(n_estimators=1000, learning_rate=1,
                           algorithm='SAMME', random_state=42)
ada_res = combo_results(feature_imp=True)

###　(10) Gradient Boost

In [ ]:
%%time
model = GradientBoostingClassifier(n_estimators=1000, random_state=42)
gb_res = combo_results(feature_imp=True)

### (11) XGBoost

In [ ]:
%%time
model = XGBClassifier(n_estimators=1000, random_state=42)
xgb_res = combo_results(feature_imp=True)

###　(12) Stacking

In [ ]:
models = [
    GaussianNB(),
    LogisticRegression(random_state=42),
    KNeighborsClassifier(),
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    AdaBoostClassifier(algorithm='SAMME', random_state=42),
    GradientBoostingClassifier(random_state=42),
    XGBClassifier(random_state=42),
]

In [ ]:
def evaluate_model(model, X=X_train_scaled, y=y_train) :
    cv = RepeatedKFold(n_splits=10, n_repeats=20, random_state=42)
    scores = cross_val_score(model, X, y, cv=10, scoring="recall")
    return scores

In [ ]:
scores = list()
names = list()
for model in models :
    score = evaluate_model(model, X_train_scaled, y_train)
    scores.append(score)
    names.append(model.__class__.__name__)
plt.figure(figsize=[10, 5])
plt.boxplot(scores, labels=names, showmeans=True)
plt.xticks(rotation=30)
plt.show()

In [ ]:
%%time
estimators = [(model.__class__.__name__, model) for model in models[1:]]
model = StackingClassifier(estimators=estimators,
                           final_estimator=GaussianNB(),
                           n_jobs=-1)
stacking_res = combo_results()

## 6. Conclusions

In [ ]:
con = pd.DataFrame({
    'Gaussian Naive Bayes': gaussian_nb_res,
    'Logistic Regression': logistic_res,
    'KNN': knn_res,
    'Decision Tree': dt_res,
    'Decision Tree (Bagging)': dt_bagging_res,
    'Random Forest': rf_res,
    'AdaBoost': ada_res,
    'Gradient Boost': gb_res,
    'XGBoost)': xgb_res,
    'Stacking': stacking_res,
})
con = np.transpose(con)
con.columns = ['Training Accuracy', 'Testing Accuracy', 'Testing F1 Score', 'Testing Sensitivity', 
               'Training Accuracy (SMOTE)', 'Testing Accuracy (SMOTE)', 'Testing F1 Score (SMOTE)', 'Testing Sensitivity (SMOTE)']
con

In [ ]:
cols = ['Training Accuracy', 'Training Accuracy (SMOTE)']
plt.plot(con.loc[:, cols], marker='o', label=['Baseline', 'SMOTE']) #
plt.legend()
plt.xticks(rotation=60)
plt.title('Training Accuracy')
plt.ylabel('Evaluation Scores')
plt.show()

In [ ]:
cols = ['Testing Sensitivity', 'Testing Sensitivity (SMOTE)']
plt.plot(con.loc[:, cols], marker='o', label=['Baseline', 'SMOTE']) #
plt.legend()
plt.xticks(rotation=60)
plt.title('Sensitivity')
plt.ylabel('Evaluation Scores')
plt.show()

In [ ]:
cols = ['Testing Accuracy', 'Testing F1 Score', 'Testing Sensitivity',
       'Testing Accuracy (SMOTE)', 'Testing F1 Score (SMOTE)', 'Testing Sensitivity (SMOTE)']
plt.plot(con.loc[:, cols], marker='o', label=['Testing Accuracy(Baseline)', 'Testing F1 Score(Baseline)',
                                              'Testing Sensitivity(Baseline)','Testing Accuracy(SMOTE)', 'Testing F1 Score(SMOTE)',
                                              'Testing Sensitivity(SMOTE)',]) 
plt.legend()
plt.xticks(rotation=60)
plt.title('Evaluation Indicators')

plt.show()